# Colorado Migration Corridor Mapper — Distro Builder

Pull the latest code from Gitea, package the app folder into a clean delivery zip,
and verify the result against the previously shipped zip.

**Run order:** just `Run All`. Each step prints what it did; nothing is destructive
(the builder refuses to overwrite an existing zip unless you set `OVERWRITE = True`).

| Step | What it does |
|---|---|
| 1 | Config — paths, exclusions, output name |
| 2 | `git pull` the latest app code |
| 3 | Collect the file list, applying exclusions |
| 4 | Preflight checks (required files present, nothing excluded leaked in) |
| 5 | Write the zip |
| 6 | Verify the zip + diff against the last delivered one |

**What is deliberately left out of the distro** (unchanged from the last delivery):
`environment_data/` and `Ext_FilesForMigrationAnalyzer.zip` (~19 GB — users download
these from Dropbox and `Setup.bat` extracts them), `app/session_data/` (per-user run
state), and dev cruft (`__pycache__`, `.git`, `.claude`, `CLAUDE*.md`, rendered `*.html`).
The bundled `python-3.13/` interpreter **is** included, verbatim, site-packages and all.

## 1 — Config

In [1]:
from pathlib import Path
import os, subprocess, sys, zipfile, fnmatch, datetime, shutil

# --- Paths -----------------------------------------------------------------
# Resolves to wherever this notebook lives, so the build works from any clone.
REPO_ROOT   = Path.cwd()
if not (REPO_ROOT / "Colorado_Migration_Mapping").is_dir():
    raise SystemExit("Run this notebook from the repo root "
                     "(the folder containing Colorado_Migration_Mapping/).")
PACKAGE_DIR = REPO_ROOT / "Colorado_Migration_Mapping" / "Colorado Migration Mapping"
OUTPUT_DIR  = Path.home() / "Desktop"

# --- Output name -----------------------------------------------------------
ZIP_BASENAME    = "Colorado Migration Mapping"   # folder name users see inside the zip
ADD_DATE_SUFFIX = True     # -> "Colorado Migration Mapping_20260730.zip"
OVERWRITE       = False    # guard: refuse to clobber an existing zip

# --- Behaviour -------------------------------------------------------------
DO_PULL       = True   # step 2: git pull before packaging
COMPRESSLEVEL = 6      # 0=store (fast, huge) .. 9=max (slow). 6 is the sweet spot.

# --- Exclusions ------------------------------------------------------------
# Paths are relative to PACKAGE_DIR and use forward slashes.

# Whole subtrees dropped outright.
EXCLUDE_DIRS = [
    "environment_data",    # ~19 GB, shipped separately via Dropbox
    "app/session_data",    # per-user run state
    ".claude",
    ".git",
    "WesternCorridorMappingTeam-main",
]

# Glob patterns matched against BOTH the relative path and the bare filename.
EXCLUDE_PATTERNS = [
    "__pycache__",  "*/__pycache__/*",
    "*.pyc", "*.pyo",
    ".gitignore", ".gitattributes",
    "CLAUDE.md", "CLAUDE.local.md", ".claude.json", ".mcp.json",
    "*.html",                              # rendered output of the .md docs
    "Ext_FilesForMigrationAnalyzer.zip",   # the 19 GB external-data bundle
    "*.zip",                               # any other stray zips
    ".ipynb_checkpoints", "*.ipynb",
    "Thumbs.db", ".DS_Store",
]

# Subtrees copied verbatim — exclusion rules are NOT applied inside these.
# (The bundled interpreter keeps its own __pycache__; stripping it only slows
#  down the user's first launch.)
VERBATIM_DIRS = ["python-3.13"]

# Files that MUST be present or the build aborts.
REQUIRED = [
    "Setup.bat",
    "Start App.bat",
    "README.txt",
    "requirements.txt",
    "INFORMATION.md",
    "app/main.py",
    "app/modules/__init__.py",
    "python-3.13/python.exe",
]

assert PACKAGE_DIR.is_dir(), f"Package dir not found: {PACKAGE_DIR}"
print("Repo:    ", REPO_ROOT)
print("Package: ", PACKAGE_DIR)
print("Output:  ", OUTPUT_DIR)

Repo:     C:\Users\leavellb\Documents\MigrationAnalyzer-1
Package:  C:\Users\leavellb\Documents\MigrationAnalyzer-1\Colorado_Migration_Mapping\Colorado Migration Mapping
Output:   C:\Users\leavellb\Desktop


## 2 — Pull the latest code

Fast-forward only, so a divergent local history stops the build instead of
producing a silent merge. Uncommitted local files (e.g. a hand-tweaked
`Setup.bat`) are reported and **are** included in the distro.

In [2]:
def git(*args, check=True):
    r = subprocess.run(["git", *args], cwd=REPO_ROOT,
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr.strip()}")
    return r.stdout.strip()


if DO_PULL:
    before = git("rev-parse", "HEAD")
    try:
        out = git("pull", "--ff-only", "origin", "main")
        print(out)
    except RuntimeError as e:
        print(e)
        print("\n--- Pull failed. Check: ---")
        print("  1. Is your VPN / tunnel to dnrapp110.naturenet.state.co.us up?")
        print("  2. Does the local branch have commits origin doesn't? "
              "(--ff-only refuses to merge)")
        raise
    after = git("rev-parse", "HEAD")

    if before == after:
        print("\nAlready up to date.")
    else:
        print("\nNew commits pulled:")
        print(git("log", "--oneline", f"{before}..{after}"))
        print("\nFiles changed:")
        print(git("diff", "--stat", before, after))
else:
    print("DO_PULL is False — packaging the working tree as-is.")

print(f"\nHEAD: {git('log', '--oneline', '-1')}")

dirty = git("status", "--porcelain")
if dirty:
    print("\n*** Uncommitted local changes — these WILL ship in the distro: ***")
    print(dirty)
else:
    print("\nWorking tree clean.")

Already up to date.

Already up to date.

HEAD: 9199022 Info cards, output fixes, UI refinements, species detection, export for bios

*** Uncommitted local changes — these WILL ship in the distro: ***
M "Colorado_Migration_Mapping/Colorado Migration Mapping/Setup.bat"
?? "Colorado_Migration_Mapping/Colorado Migration Mapping/README.txt"
?? build_distro.ipynb


## 3 — Collect the file list

Walks the package folder and applies the exclusion rules. Nothing is copied —
the zip is written straight from the source tree, so no 850 MB staging round-trip.

In [3]:
def is_excluded(rel: str, name: str) -> bool:
    """rel: forward-slash path relative to PACKAGE_DIR. name: basename."""
    for d in EXCLUDE_DIRS:
        if rel == d or rel.startswith(d + "/"):
            return True
    for pat in EXCLUDE_PATTERNS:
        if fnmatch.fnmatch(name, pat) or fnmatch.fnmatch(rel, pat):
            return True
    return False


def collect(package_dir: Path):
    """Return (kept_files, skipped_paths) as (relpath, size) / relpath lists."""
    kept, skipped = [], []

    for root, dirnames, filenames in os.walk(package_dir):
        rel_root = Path(root).relative_to(package_dir).as_posix()
        rel_root = "" if rel_root == "." else rel_root

        # Inside a verbatim subtree: take everything, no filtering.
        if any(rel_root == v or rel_root.startswith(v + "/") for v in VERBATIM_DIRS):
            for f in filenames:
                r = f"{rel_root}/{f}"
                kept.append((r, (Path(root) / f).stat().st_size))
            continue

        # Prune excluded directories in place so os.walk doesn't descend.
        pruned = []
        for d in list(dirnames):
            r = f"{rel_root}/{d}" if rel_root else d
            if r in VERBATIM_DIRS:
                continue          # handled on the next iteration, unfiltered
            if is_excluded(r, d):
                pruned.append(r)
                dirnames.remove(d)
        skipped.extend(pruned)

        for f in filenames:
            r = f"{rel_root}/{f}" if rel_root else f
            if is_excluded(r, f):
                skipped.append(r)
            else:
                kept.append((r, (Path(root) / f).stat().st_size))

    kept.sort()
    return kept, sorted(skipped)


def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:,.1f} {unit}" if unit != "B" else f"{n:,} B"
        n /= 1024


files, skipped = collect(PACKAGE_DIR)
total = sum(s for _, s in files)

print(f"{len(files):,} files, {human(total)} uncompressed\n")

print("Excluded:")
for s in skipped:
    print("  -", s)

# Size breakdown by top-level entry.
print("\nIncluded, by top-level entry:")
by_top = {}
for r, s in files:
    top = r.split("/")[0] if "/" in r else r
    by_top[top] = by_top.get(top, 0) + s
for k, v in sorted(by_top.items(), key=lambda kv: -kv[1]):
    print(f"  {human(v):>12}  {k}")

20,208 files, 802.0 MB uncompressed

Excluded:
  - .gitignore
  - Ext_FilesForMigrationAnalyzer.zip
  - app/modules/__pycache__
  - app/session_data
  - environment_data

Included, by top-level entry:
      801.0 MB  python-3.13
      806.2 KB  app
      259.9 KB  INFORMATION.md
        4.4 KB  Setup.bat
        2.0 KB  README.txt
        1.5 KB  Start App.bat
         568 B  requirements.txt


## 4 — Preflight checks

In [4]:
kept_set = {r for r, _ in files}
problems = []

missing = [r for r in REQUIRED if r not in kept_set]
if missing:
    problems.append("Missing required files: " + ", ".join(missing))

# Nothing that was supposed to be excluded should have slipped through
# (outside the verbatim interpreter tree).
def in_verbatim(r):
    return any(r == v or r.startswith(v + "/") for v in VERBATIM_DIRS)

leaked = [r for r in kept_set
          if not in_verbatim(r) and is_excluded(r, r.split("/")[-1])]
if leaked:
    problems.append("Excluded files leaked into the build: " + ", ".join(leaked[:10]))

# Catch an accidental multi-GB include.
huge = [(r, s) for r, s in files if s > 500 * 1024**2]
if huge:
    problems.append("Unexpectedly large files: " +
                    ", ".join(f"{r} ({human(s)})" for r, s in huge))

if problems:
    for p in problems:
        print("FAIL:", p)
    raise SystemExit("Preflight failed — fix the above before packaging.")

print("Preflight OK")
for r in REQUIRED:
    print("  present:", r)

Preflight OK
  present: Setup.bat
  present: Start App.bat
  present: README.txt
  present: requirements.txt
  present: INFORMATION.md
  present: app/main.py
  present: app/modules/__init__.py
  present: python-3.13/python.exe


## 5 — Write the zip

In [5]:
stamp    = datetime.date.today().strftime("%Y%m%d")
zip_name = f"{ZIP_BASENAME}_{stamp}.zip" if ADD_DATE_SUFFIX else f"{ZIP_BASENAME}.zip"
zip_path = OUTPUT_DIR / zip_name

if zip_path.exists() and not OVERWRITE:
    raise SystemExit(
        f"{zip_path} already exists.\n"
        "Set OVERWRITE = True in step 1, or change ZIP_BASENAME / ADD_DATE_SUFFIX."
    )

free = shutil.disk_usage(OUTPUT_DIR).free
if free < total * 0.5:
    print(f"WARNING: only {human(free)} free on the output drive.")

print(f"Writing {zip_path}")
print(f"  {len(files):,} files, {human(total)} -> compressing (level {COMPRESSLEVEL})\n")

step = max(1, len(files) // 20)
done = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED,
                     compresslevel=COMPRESSLEVEL) as z:
    for rel, size in files:
        z.write(PACKAGE_DIR / rel, arcname=f"{ZIP_BASENAME}/{rel}")
        done += 1
        if done % step == 0 or done == len(files):
            pct = 100 * done / len(files)
            print(f"  {pct:5.1f}%  ({done:,}/{len(files):,})", flush=True)

out_size = zip_path.stat().st_size
print(f"\nDone: {zip_path}")
print(f"  {human(out_size)}  ({100 * out_size / total:.0f}% of uncompressed)")

Writing C:\Users\leavellb\Desktop\Colorado Migration Mapping_20260730.zip
  20,208 files, 802.0 MB -> compressing (level 6)



    5.0%  (1,010/20,208)


   10.0%  (2,020/20,208)


   15.0%  (3,030/20,208)


   20.0%  (4,040/20,208)


...

   90.0%  (18,180/20,208)


   95.0%  (19,190/20,208)


  100.0%  (20,200/20,208)


  100.0%  (20,208/20,208)



Done: C:\Users\leavellb\Desktop\Colorado Migration Mapping_20260730.zip
  263.1 MB  (33% of uncompressed)


## 6 — Verify + diff against the last delivery

Reopens the finished zip to confirm it is readable and complete, then compares it
to the previously shipped zip so you can see exactly what changed in this build.

In [6]:
with zipfile.ZipFile(zip_path) as z:
    bad = z.testzip()
    if bad is not None:
        raise SystemExit(f"Zip is corrupt at: {bad}")
    names = [n for n in z.namelist() if not n.endswith("/")]
    sizes = {i.filename: i.file_size for i in z.infolist() if not i.is_dir()}

print(f"Zip verified: {len(names):,} files readable\n")
print("Top level of the delivered folder:")
for n in sorted(names):
    parts = n.split("/")
    if len(parts) == 2:
        print(f"   {human(sizes[n]):>10}  {parts[1]}")
    elif len(parts) == 3 and parts[1] == "app":
        print(f"   {human(sizes[n]):>10}  app/{parts[2]}")

# --- Diff vs the most recent previous zip ---------------------------------
prev = sorted(
    (p for p in OUTPUT_DIR.glob(f"{ZIP_BASENAME}*.zip") if p != zip_path),
    key=lambda p: p.stat().st_mtime,
)
if not prev:
    print("\nNo previous delivery found to diff against.")
else:
    p = prev[-1]
    print(f"\n--- Diff vs {p.name} "
          f"({datetime.datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M}) ---")
    with zipfile.ZipFile(p) as z:
        old = {i.filename: i.file_size for i in z.infolist() if not i.is_dir()}

    strip = lambda d: {n.split("/", 1)[1]: s for n, s in d.items() if "/" in n}
    new_s, old_s = strip(sizes), strip(old)

    added   = sorted(set(new_s) - set(old_s))
    removed = sorted(set(old_s) - set(new_s))
    changed = sorted(n for n in set(new_s) & set(old_s) if new_s[n] != old_s[n])

    def show(label, items, fmt):
        print(f"\n{label}: {len(items)}")
        for n in items[:25]:
            print("   ", fmt(n))
        if len(items) > 25:
            print(f"    ... and {len(items) - 25} more")

    show("Added",   added,   lambda n: f"+ {n} ({human(new_s[n])})")
    show("Removed", removed, lambda n: f"- {n} ({human(old_s[n])})")
    show("Changed", changed,
         lambda n: f"~ {n} ({human(old_s[n])} -> {human(new_s[n])})")

    print(f"\nSize: {human(p.stat().st_size)} -> {human(out_size)}")

print(f"\n\nREADY TO DELIVER:  {zip_path}")

Zip verified: 20,208 files readable

Top level of the delivered folder:
     259.9 KB  INFORMATION.md
       2.0 KB  README.txt
       4.4 KB  Setup.bat
       1.5 KB  Start App.bat
     490.1 KB  app/main.py
        568 B  requirements.txt

--- Diff vs Colorado Migration Mapping.zip (2026-07-24 13:10) ---

Added: 0

Removed: 1
    - .gitignore (372 B)

Changed: 5
    ~ app/assets/style.css (5.9 KB -> 5.7 KB)
    ~ app/main.py (451.6 KB -> 490.1 KB)
    ~ app/modules/data_ingestion.py (45.1 KB -> 47.6 KB)
    ~ app/modules/modeling.py (87.6 KB -> 89.8 KB)
    ~ app/modules/population_outputs.py (86.8 KB -> 92.1 KB)

Size: 265.1 MB -> 263.1 MB


READY TO DELIVER:  C:\Users\leavellb\Desktop\Colorado Migration Mapping_20260730.zip
